In [ ]:
import os
import pathlib
import zipfile
import pandas as pd
import numpy as np
import tensorflow as tf
tf.__version__

In [ ]:
# From challenge author repo
def package_predictions_for_submission(scores, threshold, output=pathlib.Path('./submission.zip')):
  
  filename_variable_pairs = {
    'threshold.txt': threshold,
    'scores.txt': scores,
  }

  # The most straightforward way to populate a zip file is with actual files on disk
  # Therefore, temporarily create each file in the working directory  
  for f, d in filename_variable_pairs.items():
    np.savetxt(f, d)

  # Create the zip file
  with zipfile.ZipFile(output, mode='w') as z:
    for f, _ in filename_variable_pairs.items():
      z.write(f)
  
  # ...the cleanup after ourselves to avoid clutter
  for f, _ in filename_variable_pairs.items():
    os.remove(f)

  print(f'Successfully created CodaBench submission file: `{output}`. Upload this zip file to CodaBench to complete your submission.')

In [ ]:
X_ref = np.load('XRef_AllStepsR_f32_norm.npy')
Y_ref = np.load('YRef_AllStepsR.npy')
X_probe = np.load('XProbe_AllStepsR_f32_norm.npy')
Y_probe = np.load('YProbe_AllStepsR.npy')

In [ ]:
import inceptionembed
model = inceptionembed.get_model(n_classes=200, input_shape=(None, 75*40*2), reduce=[tf.keras.layers.GlobalAveragePooling1D()])
# model.load_weights('081_checkpoint_epoch_200.keras') # 028
# model.load_weights('081_checkpoint_epoch_150.keras') # 031
# model.load_weights('082_checkpoint_epoch_150.keras') # 035
# model.load_weights('082_checkpoint_epoch_100.keras') # 036
model.load_weights('082_checkpoint_epoch_50.keras') # 037

In [ ]:
embedding_model = tf.keras.Model(
    inputs=model.input,
    outputs=model.get_layer("global_average_pooling1d_1").output
)

#### 1D architecture with both foot, concat on series channel.

In [ ]:
X_ = X_ = np.zeros((75, 101, 75, 40, 2), dtype=np.float32)
for i in range(75):
    X_[i, :, :, :, 0] = X_ref[2*i]
    X_[i, :, :, :, 1] = X_ref[2*i+1]
    """
    # inverted (030)
    X_[i, :, :, :, 1] = X_ref[2*i][:, :, ::-1]
    X_[i, :, :, :, 0] = X_ref[2*i+1][:, :, ::-1]
    """

X_ref.shape, X_.shape

In [ ]:
X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
ref_emb = embedding_model.predict(X_)
ref_emb.shape

In [ ]:
probe_emb = []
for j in range(10):
    X_ = X_ = np.zeros((1000, 101, 75, 40, 2), dtype=np.float32)
    for i in range(1000):
        X_[i, :, :, :, 0] = X_probe[j*2000+2*i]
        X_[i, :, :, :, 1] = X_probe[j*2000+2*i+1]
        """
        # inverted (029)
        X_[i, :, :, :, 1] = X_probe[j*2000+2*i]
        X_[i, :, :, :, 0] = X_probe[j*2000+2*i+1]
        # inverted (030)
        X_[i, :, :, :, 1] = X_probe[j*2000+2*i][:, :, ::-1]
        X_[i, :, :, :, 0] = X_probe[j*2000+2*i+1][:, :, ::-1]
        """
    X_ = X_.reshape(X_.shape[0], X_.shape[1], -1)
    probe_emb.append(embedding_model.predict(X_[0:1000], batch_size=64))

In [ ]:
probe_emb = np.concatenate(probe_emb, axis=0)
probe_emb.shape

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

# Archives

## submission_035 & 036 & 037

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb#.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb#.reshape(-1,2,128).mean(axis=1)

In [ ]:
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []
for c in classes:
    members = ref_person[Y_ref[:, 0] == c]
    
    # On garde ce qui est dans le 2xstd de l'average de l'embedding
    centroid = members.mean(axis=0, keepdims=True)
    dists = cosine_similarity(members, centroid)[:, 0]
    mask = dists > (dists.mean() - 2 * dists.std())
    members = members[mask]
    
    # Why not medoid : "sample closest to the centroid"
    centroid = members.mean(axis=0, keepdims=True)
    medoid_idx = cosine_similarity(members, centroid)[:, 0].argmax()
    prototypes.append(members[medoid_idx])
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

print(f"Raw scores — min: {scores.min():.4f}, max: {scores.max():.4f}, mean: {scores.mean():.4f}")

In [ ]:
# z-score normalization par classe
classes_probe = np.unique(Y_probe[:, 0])
class_score_stats = {}

for c in classes_probe:
    mask = Y_probe[:, 0] == c
    class_scores = scores[mask]
    class_score_stats[c] = {
        'mean': class_scores.mean(),
        'std': class_scores.std()
    }

normalized_scores = np.array([
    (scores[i] - class_score_stats[Y_probe[i, 0]]['mean']) / (class_score_stats[Y_probe[i, 0]]['std'] + 1e-8)
    for i in range(len(scores))
])

normalized_scores -= normalized_scores.min()

print(f"Normalized scores — min: {normalized_scores.min():.4f}, max: {normalized_scores.max():.4f}, mean: {normalized_scores.mean():.4f}")

In [ ]:
threshold = np.array([3.0])
pred_known = normalized_scores > threshold
pred_known.mean()

In [ ]:
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_035.zip'))
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_036.zip'))
package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_037.zip'))

## submission_028 & 029 & 030 & 031

In [ ]:
ref_emb = tf.math.l2_normalize(ref_emb, axis=1).numpy()
probe_emb = tf.math.l2_normalize(probe_emb, axis=1).numpy()

In [ ]:
ref_person = ref_emb#.reshape(-1,2,128).mean(axis=1)
probe_person = probe_emb#.reshape(-1,2,128).mean(axis=1)

In [ ]:
classes = np.unique(Y_ref)

prototypes = []
proto_labels = []
for c in classes:
    members = ref_person[Y_ref[:, 0] == c]
    
    # On garde ce qui est dans le 2xstd de l'average de l'embedding
    centroid = members.mean(axis=0, keepdims=True)
    dists = cosine_similarity(members, centroid)[:, 0]
    mask = dists > (dists.mean() - 2 * dists.std())
    members = members[mask]
    
    # Why not medoid : "sample closest to the centroid"
    centroid = members.mean(axis=0, keepdims=True)
    medoid_idx = cosine_similarity(members, centroid)[:, 0].argmax()
    prototypes.append(members[medoid_idx])
    proto_labels.append(c)

prototypes = np.array(prototypes)
proto_labels = np.array(proto_labels)

In [ ]:
# score sur la class claim
scores = []
for i in range(len(probe_person)):
    claim = Y_probe[i]

    idx = np.where(proto_labels == claim)[0][0]

    score = cosine_similarity(
        probe_person[i:i+1],
        prototypes[idx:idx+1]
    )[0,0]

    scores.append(score)

scores = np.array(scores)

print(f"Raw scores — min: {scores.min():.4f}, max: {scores.max():.4f}, mean: {scores.mean():.4f}")

In [ ]:
# z-score normalization par classe
classes_probe = np.unique(Y_probe[:, 0])
class_score_stats = {}

for c in classes_probe:
    mask = Y_probe[:, 0] == c
    class_scores = scores[mask]
    class_score_stats[c] = {
        'mean': class_scores.mean(),
        'std': class_scores.std()
    }

normalized_scores = np.array([
    (scores[i] - class_score_stats[Y_probe[i, 0]]['mean']) / (class_score_stats[Y_probe[i, 0]]['std'] + 1e-8)
    for i in range(len(scores))
])

normalized_scores -= normalized_scores.min()

print(f"Normalized scores — min: {normalized_scores.min():.4f}, max: {normalized_scores.max():.4f}, mean: {normalized_scores.mean():.4f}")

In [ ]:
threshold = np.array([3.2])
pred_known = normalized_scores > threshold
pred_known.mean()

In [ ]:
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_028-2.zip'))
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_029.zip'))
# package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_030.zip'))
package_predictions_for_submission(normalized_scores, threshold, output=pathlib.Path('./submission_031.zip'))